## Формируем мегатаблицу для обучения модели

Основные (выстрелившие) поля:
- `city_size` - Размер города покупателя
- `city_center_distance_km` - Удалённость покупателя от центра города
- `city_center_level` - Удалённость покупателя от центра города (относительная)
- `days_after_order` - Количество дней, прошедшее после последней покупки
- `state_income` - Средний доход по штату
- `last_delivery_distance_km` - Дальность последней доставки в километрах
- `last_delivery_cost` - Цена последней доставки
- `last_order_cost` - Цена последней покупки
- `last_review_score` - Оценка в последнем отзыве
- `last_review_length` - Длина последнего отзыва
- `last_review_product_state` - Состояние продукта по последнему отзыву (NLP анализ)
- `last_review_sentiment` - Настроение клиента по последнему отзыву (NLP анализ)
- `avg_order_interval_days` - Средний интервал между прошлыми покупками
- `main_payment_type` - Основной способ оплаты
- `last_order_day_of_year` - Номер дня в году для последнего ордера (учёт сезонности)
- `percent_null_reviews` - Доля продаж без отзыва у продавца
- `dispersion_review_score` - Дисперсия оценки отзывов продавца
- `avg_review_message_length` - Средняя длина отзывов у продавца
- `product_count` - Количество продуктов у продавца
- `order_date_variance` - Дисперсия продаж у продавца
- `product_name_lenght` - Длина названия последнего купленного продукта
- `product_description_lenght` - Длина описания последнего купленного продукта
- `product_photos_qty` - Количество фотографий у последнего купленного продукта`


In [1]:
import json
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from dotenv import load_dotenv
from catboost import CatBoostClassifier
from sqlalchemy import create_engine, text
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_sample_weight

print('Необходимые библиотеки загружены.')

Необходимые библиотеки загружены.


In [1]:
# загрузка переменных из .env
load_dotenv()

# получаем токен из окружения
POSTGRES_CONNECTION_URL = os.getenv("POSTGRES_CONNECTION_URL")

try:
    # 1. Загрузка данных
    engine = create_engine(POSTGRES_CONNECTION_URL)

except Exception as e:
    print(f"An error occurred: {e}")

finally:
    # engine.dispose()
    print("Connected.")

Done.


In [3]:
try:
    with engine.connect() as connection:
        connection.execute(text("""
CREATE TABLE customer_features_timeline (
    id SERIAL PRIMARY KEY, -- Уникальный идентификатор записи
    timeline_date DATE, -- Дата расчёта параметров. Идём день за днём.
    order_id UUID, -- Уникальный идентификатор ордера
    customer_unique_id UUID, -- Уникальный идентификатор клиента
    customer_state TEXT, -- Штат клиента
    customer_city TEXT, -- Город клиента
    city_size REAL, -- Размер города
    city_center_distance_km REAL, -- Расстояние от центра города в км
    city_center_level REAL, -- Расстояние от центра города
    days_after_order INT, -- Сколько прошло дней с последнего заказа
    days_before_order INT, -- ТАРГЕТ. Сколько осталось дней до следующего заказа
    is_customer_will_order boolean, -- ТАРГЕТ. Он ещё будет покупать в будущем?
    state_density REAL, -- Плотность населения по штату
    state_income REAL, -- Средний доход по штату
    state_north_south_index INT, -- Северный или южный штат. 0 - юг, 9 - север.
    last_delivery_distance_km REAL, -- Последний заказ: расстояние между продавцом и покупателем
    last_delivery_delay_days INT, -- Последний заказ: количество дней задержки
    last_delivery_actual_days INT null, -- Последний заказ: количество дней реальной доставки
    last_delivery_cost REAL, -- Последний заказ: цена доставки
    last_order_cost REAL, -- Последний заказ: цена ордера
    last_cancelled boolean, -- Последний заказ: ордер был отменён
    last_delivered boolean, -- Последний заказ: заказ доставлен
    last_review_score INT, -- Последний отзыв: оценка клиента
    last_review_length INT, -- Последний отзыв: длина коментария в символах
    last_review_match_description boolean, -- Последний отзыв: товар соответствует описанию?
    last_review_product_state INT, -- Последний отзыв: состояние товара
    last_review_recommend boolean, -- Последний отзыв: клиент порекомендует сервис?
    last_review_sentiment INT, -- Последний отзыв: какое настроение клиента от -2 до 2?
    last_review_buy_again boolean, -- Последний отзыв: клиент обещает покупать снова?
    last_review_want_leave boolean, -- Последний отзыв: клиент угрожает уйти?
    last_review_unexpected_issues boolean, -- Последний отзыв: были неожиданные проблемы?
    last_review_quality_happy boolean, -- Последний отзыв: доволен качеством?
    avg_order_interval_days INT, -- Средний интервал между заказами
    main_payment_type TEXT, -- Предпочитаемый способ оплаты
    last_product_rating_min REAL, -- Последний заказ: средний рейтинг худшего товара
    last_product_rating_max REAL, -- Последний заказ: средний рейтинг лудшего товара
    last_product_size_max TEXT, -- Размер продукта максимальный
    last_product_weight_max REAL, -- Вес продукта максимальный
    last_product_name_lenght INT,
    last_product_description_lenght INT,
    last_product_photos_qty INT,
    first_product_abc REAL, -- ABC категория, первый купленный товар
    first_product_xyz REAL, -- XYZ категория, первый купленный товар
    last_product_abc REAL, -- ABC категория, последний купленный товар A = 1, B = 2, C = 3
    last_product_xyz REAL, -- XYZ категория, последний купленный товар X = 1, Y = 2, Z = 3
    last_order_month INT, -- Месяц последнего заказа
    last_order_day_of_year INT, -- Номер дня в году последнего заказа
    seller_orders_min INT, -- Продавец: общее количество ордеров у самого скромного продавца
    seller_repeat_rate_min REAL, -- Продавец: доля повторных покупок у худшего продавца
    seller_repeat_rate_max REAL, -- Продавец: доля повторных покупок у лучшего продавца
    seller_cancel_rate_min REAL, -- Продавец: доля отменённых заказов у худшего продавца
    rfm_score TEXT -- RFM клиента
);"""))
        connection.commit()
    print("Таблица 'customer_features_timeline' успешно создана (если её не было).")
except Exception as e:
    print(f"Ошибка при создании таблицы: {e}")

Таблица 'customer_features_timeline' успешно создана (если её не было).
